# 07 · Ablation study

Stepwise configurations actually implemented and executed: B0 vector-only → E1 hybrid (SQL RRF) → E2 improved hybrid → E2S E2 + evidence selection. Reranking (E3) is not implemented and is not reported. Absolute (percentage points) and relative differences with 95% cluster-bootstrap intervals.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
frames = {}
for target in ("local", "production"):
    for ds in ("heldout_v0.2", "pilot_v0.1"):
        f = A.RESULTS / f"retrieval_{target}_{ds}.csv"
        if target == "production":   # erratum 2026-11-23: announcement ids corrected (scripts/rescore_production.py)
            f = A.RESULTS / f"retrieval_{target}_{ds}_rescored.csv"
        if f.exists():
            frames[(target, ds)] = A.load_retrieval(f.name)
avail = pd.DataFrame([(t, d, len(df), df["method"].nunique(), int((df["error"].fillna("") != "").sum()))
                      for (t, d), df in frames.items()], columns=["target", "dataset", "rows", "methods", "failed requests"])
avail

In [ ]:
steps = [("E1", "B0", "hybrid fusion (equal-weight SQL RRF) vs vector-only"),
         ("E2", "B0", "improved hybrid vs vector-only"),
         ("E2", "E1", "weighted fusion + query analysis vs equal-weight RRF"),
         ("E2S", "E2", "assistant evidence selection on top of E2 (no query rewriting)")]
rows = []
for (t, d), df in frames.items():
    for a, b, what in steps:
        if {a, b} <= set(df["method"]):
            for col, name in (("r1", "Recall@1"), ("r5", "Recall@5"), ("rr", "MRR"), ("ndcg5", "nDCG@5")):
                r = A.paired_difference(df, a, b, col)
                rows.append(dict(target=t, dataset=d, step=f"{a} vs {b}", what=what, metric=name, n=r["n"],
                                 baseline=r["mean_b"], variant=r["mean_a"], diff_pp=100 * r["diff"],
                                 lo_pp=100 * r["lo"], hi_pp=100 * r["hi"], relative_pct=r["relative_pct"],
                                 ci_excludes_zero=r["ci_excludes_zero"]))
abl = pd.DataFrame(rows)
abl.to_csv(A.RESULTS / "ablation.csv", index=False)
abl[abl.dataset == "heldout_v0.2"].round(3)

In [ ]:
sub = abl[(abl.dataset == "heldout_v0.2") & (abl.metric.isin(["Recall@1", "MRR"]))]
STEP_ORDER = ["E1 vs B0", "E2 vs B0", "E2 vs E1", "E2S vs E2"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))   # own y-axis per panel: the targets ran different comparisons
for ax, target in zip(axes, ["local", "production"]):
    s = sub[sub.target == target]
    if s.empty:
        ax.text(0.5, 0.5, "Not evaluated", ha="center", transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([]); continue
    labels = [st for st in STEP_ORDER if st in set(s["step"])]
    for j, (metric, color) in enumerate((("Recall@1", A.COLORS["B0"]), ("MRR", A.COLORS["E1"]))):
        m = s[s.metric == metric].set_index("step").loc[labels]
        y = np.arange(len(labels)) + (j - 0.5) * 0.32
        ax.errorbar(m["diff_pp"], y, xerr=[m["diff_pp"] - m["lo_pp"], m["hi_pp"] - m["diff_pp"]], fmt="o",
                    color=color, ecolor=color, elinewidth=1.6, capsize=3, markersize=7, label=metric)
        for xv, yv in zip(m["diff_pp"], y):
            ax.text(xv, yv + 0.13, f"{xv:+.1f}", ha="center", fontsize=8, color=A.INK)
    ax.axvline(0, color=A.INK2, linewidth=1)
    ax.set_yticks(np.arange(len(labels)), labels); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.set_ylim(-0.6, len(labels) - 0.15)     # headroom so the top value labels clear the title
    ax.set_xlabel("difference, percentage points (95% CI)")
    ax.set_title(f"{target.capitalize()} · held-out v0.2 (n = {int(s['n'].iloc[0])})")
handles, labels = next((ax.get_legend_handles_labels() for ax in axes if ax.get_legend_handles_labels()[0]), ([], []))
fig.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.07))
fig.tight_layout(); print(A.save(fig, "07_ablation_differences"))

In [ ]:
h = abl[(abl.dataset == "heldout_v0.2") & (abl.metric == "Recall@1")]
lines = [f"- {r.target}, {r.step}: {r.diff_pp:+.1f} pp ({r.lo_pp:+.1f} to {r.hi_pp:+.1f}), relative {r.relative_pct:+.0f}%"
         + (" - interval excludes 0" if r.ci_excludes_zero else " - interval includes 0") for r in h.itertuples()]
display(Markdown("**Recall@1 on the held-out set:**\n\n" + "\n".join(lines) + "\n\nOnly intervals that exclude 0 "
                 "support a claim of difference; the others are inconclusive at this sample size."))